In [ ]:
import torch
import triton
import triton.language as tl
import time

In [2]:
def is_cuda():
    return triton.runtime.driver.active.get_current_target().backend == "cuda"

In [3]:
def is_hip_mi200():
    target = triton.runtime.driver.active.get_current_target()
    return target.backend == 'hip' and target.arch == 'gfx90a'

In [ ]:
"""
MatMul+Relu+Add Fused Optimization.
The kernel uses several optimization techniques:

  1. Shared memory tiling.
  2. Register tiling.
  3. Cooperative fetching.
  4. Operator Fusion
  5. Write cache / epilogue fusion.

"""

# -----------------------------------------------------------------------------
# Tiling parameters - May need to change these to achieve better results.
# Tuned for Colab T4 (2048x2048 fp16). Re-run the grid-search cell after
# changing these values.
# -----------------------------------------------------------------------------
BLOCK_M = 128  # Tile size in the M dimension.
BLOCK_N = 128  # Tile size in the N dimension.
BLOCK_K = 32   # Tile size in the K dimension.


# -----------------------------------------------------------------------------
# Triton Kernel: Matrix Multiplication + ReLU + Add
#
# The kernel uses:
#   Step 1: Tile assignment (each kernel computes a tile of C)
#   Step 2: Shared memory tiling + Cooperative Fetching: Load tiles of A and B.
#   Step 3: Register tiling: Use a register accumulator.
#   Step 4: Add and ReLU fusion
#   Step 5: Write cache/Epilogue: Write the final tile back to global memory.
# -----------------------------------------------------------------------------
@triton.jit
def matmul_add_relu_kernel_fp16(
    a_ptr, b_ptr, c_ptr, d_ptr,
    M: tl.constexpr, N: tl.constexpr, K: tl.constexpr,
    stride_am: tl.constexpr, stride_ak: tl.constexpr,
    stride_bk: tl.constexpr, stride_bn: tl.constexpr,
    stride_cm: tl.constexpr, stride_cn: tl.constexpr,
    stride_dm: tl.constexpr, stride_dn: tl.constexpr,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    # -------------------------------------------------------------------------
    # Step 1: Tile Assignment
    #
    # Each kernel instance is mapped to a tile in the output matrix D.
    # Compute the starting indices (m_start, n_start) for this tile.
    # -------------------------------------------------------------------------
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    m_start = pid_m * BLOCK_M
    n_start = pid_n * BLOCK_N

    offs_m = m_start + tl.arange(0, BLOCK_M)
    offs_n = n_start + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    # -------------------------------------------------------------------------
    # Step 2: Register Tiling
    # Accumulate in fp16 as required; tl.dot still uses tensor cores on T4.
    # -------------------------------------------------------------------------
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float16)

    # Pointers to the first K-tile of A (BLOCK_M x BLOCK_K) and B (BLOCK_K x BLOCK_N).
    a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn

    # -------------------------------------------------------------------------
    # Step 3: Shared Memory Tiling & Cooperative Fetching.
    # Load successive K-tiles of A and B, masking OOB accesses.
    # -------------------------------------------------------------------------
    for k in range(0, K, BLOCK_K):
        k_offs = k + offs_k
        a_mask = (offs_m[:, None] < M) & (k_offs[None, :] < K)
        b_mask = (k_offs[:, None] < K) & (offs_n[None, :] < N)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)
        acc += tl.dot(a, b).to(tl.float16)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # -------------------------------------------------------------------------
    # Step 4: Apply ReLU and Add C to the accumulator
    # D = ReLU(A @ B + C)
    # -------------------------------------------------------------------------
    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    c = tl.load(c_ptrs, mask=c_mask, other=0.0)
    acc = acc + c
    acc = tl.maximum(acc, 0.0)

    # -------------------------------------------------------------------------
    # Step 5: Write Cache / Epilogue Fusion: Write the computed tile to D.
    # -------------------------------------------------------------------------
    d_ptrs = d_ptr + offs_m[:, None] * stride_dm + offs_n[None, :] * stride_dn
    tl.store(d_ptrs, acc, mask=c_mask)


In [62]:
def matmul_add_relu_fp16(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """
    Computes Output = ReLU(A @ B + C) using fp16 precision for maximum throughput.
    """
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, "Incompatible dimensions"

    d = torch.empty((M, N), device=a.device, dtype=torch.float16)
    # Create launch grid
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))

    matmul_add_relu_kernel_fp16[grid](
        a, b, c, d,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        d.stride(0), d.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        num_warps=8,
        num_stages=4,
    )
    return d

In [68]:
# Reference implementation using PyTorch
def reference_matmul_add_relu(A, B, C):
    result = torch.matmul(A, B).add(C).relu_()
    return result

In [ ]:
# -----------------------------------------------------------------------------
# Accuracy Tests
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    torch.manual_seed(0)
    a = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    b = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    c = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    triton_output = matmul_add_relu_fp16(a, b, c)
    torch_output = reference_matmul_add_relu(a, b, c)
    print(f"triton_output_with_fp16_inputs={triton_output}")
    print(f"torch_output_with_fp16_inputs={torch_output}")
    rtol = 1e-2 if is_hip_mi200() else 0.032
    if torch.allclose(triton_output, torch_output, atol=0.15, rtol=rtol):
        print("✅ Triton and Torch match")
    else:
        diff = triton_output - torch_output
        abs_diff = torch.abs(diff)
        max_abs_diff = torch.max(abs_diff)
        print(f"❌ Triton and Torch differ: {max_abs_diff=}")

In [ ]:
# -----------------------------------------------------------------------------
# Performance Benchmark 
# IMPORTANT: DO NOT CHANGE THIS CODE. 
# ANY CHANGES TO THIS CODE (INCLUDING DIMENSIONS, REPEATS, etc.) CAN CAUSE DIFFERENT SPEEDUP RESULTS.
# -----------------------------------------------------------------------------
M = 2048
K = 2048
N = 2048

# KEEP THESE MATRICES IN FP16. FP32 WILL NOT PROVIDE ACCURATE RESULTS
A = torch.randn((M, K), device="cuda", dtype=torch.float16)
B = torch.randn((K, N), device="cuda", dtype=torch.float16)
C = torch.randn((M, N), device="cuda", dtype=torch.float16)

# warmup
_ = matmul_add_relu_fp16(A, B, C)
_ = reference_matmul_add_relu(A, B, C)

REPEATS = 5000

# time the triton implementation
print("Triton implementation")
torch.cuda.synchronize()
start = time.perf_counter()
for _ in range(REPEATS):
    _ = matmul_add_relu_fp16(A, B, C)
torch.cuda.synchronize()
triton_time = (time.perf_counter() - start) / REPEATS

# time pytorch
print("PyTorch implementation")
torch.cuda.synchronize()
start = time.perf_counter()
for _ in range(REPEATS):
    _ = reference_matmul_add_relu(A, B, C)
torch.cuda.synchronize()
torch_time = (time.perf_counter() - start) / REPEATS

print(f"Performance comparison for matrix multiplication ({M}x{K} @ {K}x{N}):")
print(f"Triton implementation: {triton_time*1000:.2f} ms")
print(f"PyTorch implementation: {torch_time*1000:.2f} ms")

print(f"\nSpeedup of Triton vs PyTorch: {torch_time/triton_time:.2f}x")

In [ ]:
# Grid search over tile sizes for 2048x2048 fp16 on T4.
# After it finishes, copy the best BLOCK_M/BLOCK_N/BLOCK_K into the kernel cell
# and re-run the kernel + benchmark cells. Keep inputs in fp16.
import itertools

def _run_kernel_once(a, b, c, block_m, block_n, block_k):
    M, K = a.shape
    _, N = b.shape
    d = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    matmul_add_relu_kernel_fp16[grid](
        a, b, c, d,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        d.stride(0), d.stride(1),
        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k,
    )
    return d

def grid_search_block_sizes(
    sizes=(32, 64, 128, 256),
    k_sizes=(16, 32, 64),
    warmup=5,
    repeats=30,
    dim=2048,
):
    a = torch.randn((dim, dim), device="cuda", dtype=torch.float16)
    b = torch.randn((dim, dim), device="cuda", dtype=torch.float16)
    c = torch.randn((dim, dim), device="cuda", dtype=torch.float16)
    ref = reference_matmul_add_relu(a, b, c)

    results = []
    for bm, bn, bk in itertools.product(sizes, sizes, k_sizes):
        # Skip configs that are too large for T4 shared memory (~64KB).
        smem_bytes = (bm * bk + bk * bn) * 2
        if smem_bytes > 48 * 1024:
            continue
        try:
            for _ in range(warmup):
                out = _run_kernel_once(a, b, c, bm, bn, bk)
            torch.cuda.synchronize()
            start = time.perf_counter()
            for _ in range(repeats):
                out = _run_kernel_once(a, b, c, bm, bn, bk)
            torch.cuda.synchronize()
            ms = (time.perf_counter() - start) / repeats * 1000
            ok = torch.allclose(out, ref, atol=0.15, rtol=0.032)
            results.append((ms, ok, bm, bn, bk))
            print(f"BLOCK_M={bm:3d} BLOCK_N={bn:3d} BLOCK_K={bk:3d}  {ms:7.3f} ms  match={ok}")
        except Exception as e:
            print(f"BLOCK_M={bm:3d} BLOCK_N={bn:3d} BLOCK_K={bk:3d}  FAILED: {e}")

    results.sort()
    print("\nBest configs:")
    for row in results[:10]:
        print(row)
    return results

# Uncomment on Colab T4 after the kernel cell has been executed:
# grid_search_block_sizes()
